El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

## Preparación de datos

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb
import warnings

In [2]:
df = pd.read_csv('/datasets/car_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

In [3]:
df.head()

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


In [4]:
df.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000


In [5]:
print('Los valores nulos son: ', df.isna().sum())

Los valores nulos son:  DateCrawled              0
Price                    0
VehicleType          37490
RegistrationYear         0
Gearbox              19833
Power                    0
Model                19705
Mileage                  0
RegistrationMonth        0
FuelType             32895
Brand                    0
NotRepaired          71154
DateCreated              0
NumberOfPictures         0
PostalCode               0
LastSeen                 0
dtype: int64


Se puede observar que las columnas "RegistrationYear" y "Power" tienen valores mínimos y máximos imposibles, esto se puede deber a que los datos sean corruptos, independientemente del origen del error, toca arreglar esos rangos.

In [6]:
print('El Dataframe sin filtrar tiene' , len(df), 'filas')

El Dataframe sin filtrar tiene 354369 filas


In [7]:
df_filtrado = df[(df['RegistrationYear'] >= 1900) & (df['RegistrationYear'] <= 2016) & (df['Power'] >= 45) & (df['Power'] <= 500)].copy()

In [8]:
print('El Dataframe filtrado tiene' , len(df_filtrado), 'filas')
print()
print('Se perdió aproximadamente un 15% de las filas en el filtrado')

El Dataframe filtrado tiene 300243 filas

Se perdió aproximadamente un 15% de las filas en el filtrado


In [9]:
df_filtrado['VehicleType'] = df_filtrado['VehicleType'].fillna('unknown')
df_filtrado['Gearbox'] = df_filtrado['Gearbox'].fillna('unknown')
df_filtrado['Model'] = df_filtrado['Model'].fillna('unknown')
df_filtrado['FuelType'] = df_filtrado['FuelType'].fillna('unknown')
df_filtrado['NotRepaired'] = df_filtrado['NotRepaired'].fillna('unknown')
df_filtrado.head(15)

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
1,24/03/2016 10:58,18300,coupe,2011,manual,190,unknown,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,unknown,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17
5,04/04/2016 17:36,650,sedan,1995,manual,102,3er,150000,10,petrol,bmw,yes,04/04/2016 00:00,0,33775,06/04/2016 19:17
6,01/04/2016 20:48,2200,convertible,2004,manual,109,2_reihe,150000,8,petrol,peugeot,no,01/04/2016 00:00,0,67112,05/04/2016 18:18
7,21/03/2016 18:54,0,sedan,1980,manual,50,other,40000,7,petrol,volkswagen,no,21/03/2016 00:00,0,19348,25/03/2016 16:47
8,04/04/2016 23:42,14500,bus,2014,manual,125,c_max,30000,8,petrol,ford,unknown,04/04/2016 00:00,0,94505,04/04/2016 23:42
9,17/03/2016 10:53,999,small,1998,manual,101,golf,150000,0,unknown,volkswagen,unknown,17/03/2016 00:00,0,27472,31/03/2016 17:17
10,26/03/2016 19:54,2000,sedan,2004,manual,105,3_reihe,150000,12,petrol,mazda,no,26/03/2016 00:00,0,96224,06/04/2016 10:45


Aquí convertí todos los valores nulos de las columnas categóricas en "Unknown", ya que sería ilogico eliminar las columnas con valores nulos solo por eso.

## Entrenamiento del modelo 

### Modelo de regresión lineal (Prueba de cordura)

In [10]:
%%time
categorical_cols = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']
df_ohe = pd.get_dummies(data=df_filtrado, columns=categorical_cols)

X = df_ohe.drop(columns=['Price', 'DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures', 'PostalCode'])
y = df_ohe['Price']

X_train_ohe, X_valid_ohe, y_train_ohe, y_valid_ohe = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train_ohe, y_train_ohe)
predict_lineal = model.predict(X_valid_ohe)
mse_lineal = mean_squared_error(y_valid_ohe, predict_lineal)
rmse_lineal = mse_lineal ** 0.5
print(f"RECM de la regresión lineal en validación: {rmse_lineal:.2f}")

RECM de la regresión lineal en validación: 2606.30
CPU times: user 11.5 s, sys: 3.64 s, total: 15.2 s
Wall time: 8.47 s


In [11]:
%%time
predict_lineal = model.predict(X_valid_ohe)

CPU times: user 44.9 ms, sys: 331 ms, total: 376 ms
Wall time: 220 ms


### Modelo Bosque aleatorio

In [12]:

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_rf = df_filtrado.drop(columns=['Price', 'DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures', 'PostalCode']).copy()
y_rf = df_filtrado['Price'].copy()

X_rf[categorical_cols] = encoder.fit_transform(X_rf[categorical_cols])

X_train_rf, X_valid_rf, y_train_rf, y_valid_rf = train_test_split(X_rf, y_rf, test_size=0.25, random_state=42)

param_grid_rf = {
    'n_estimators': [20, 50, 100],
    'max_depth': [5, 10, 15]
}

In [13]:
%%time
rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1
)

rf_grid.fit(X_train_rf, y_train_rf)

best_rf = rf_grid.best_estimator_
predict_best_rf = best_rf.predict(X_valid_rf)
mse_best_rf = mean_squared_error(y_valid_rf, predict_best_rf)
rmse_best_rf = mse_best_rf ** 0.5

print(f"Mejores hiperparámetros RF: {rf_grid.best_params_}")
print(f"Mejor RECM de Bosque Aleatorio en validación: {rmse_best_rf:.2f}")


Mejores hiperparámetros RF: {'max_depth': 15, 'n_estimators': 100}
Mejor RECM de Bosque Aleatorio en validación: 1695.17
CPU times: user 36.8 s, sys: 132 ms, total: 36.9 s
Wall time: 2min 37s


In [14]:
model_rf_final = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)


In [15]:
%%time
model_rf_final.fit(X_train_rf, y_train_rf)

predict_rf_final = model_rf_final.predict(X_valid_rf)
mse_rf_final = mean_squared_error(y_valid_rf, predict_rf_final)
rmse_rf_final = mse_rf_final ** 0.5

print(f"RECM final de Bosque Aleatorio en validación: {rmse_rf_final:.2f}")

RECM final de Bosque Aleatorio en validación: 1695.17
CPU times: user 37 s, sys: 0 ns, total: 37 s
Wall time: 18.6 s


In [16]:
%%time
predict_rf_final = model_rf_final.predict(X_valid_rf)

CPU times: user 1e+03 ms, sys: 0 ns, total: 1e+03 ms
Wall time: 509 ms


### Modelo LightGBM

In [17]:

X = df_filtrado.drop(columns=['Price', 'DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures', 'PostalCode'])
y = df_filtrado['Price']


X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

X_train = X_train.copy()
X_valid = X_valid.copy()

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

In [31]:
%%time
param_sets = [
    {'n_estimators': 50, 'learning_rate': 0.1, 'num_leaves': 31},
    {'n_estimators': 100, 'learning_rate': 0.1, 'num_leaves': 31},
    {'n_estimators': 100, 'learning_rate': 0.05, 'num_leaves': 63}
]

best_rmse = float('inf')
best_lgb_model = None

warnings.filterwarnings('ignore', category=UserWarning)

for i, params in enumerate(param_sets, 1):
    model = lgb.LGBMRegressor(**params, random_state=42)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

    preds = model.predict(X_valid)
    rmse = mean_squared_error(y_valid, preds) ** 0.5

    print(f"Conjunto {i} {params} -> RECM: {rmse:.2f}")

    if rmse < best_rmse:
        best_rmse = rmse
        best_lgb_model = model

print(f"\nMejor RECM alcanzado con LightGBM: {best_rmse:.2f}")

[1]	valid_0's l2: 1.84147e+07
[2]	valid_0's l2: 1.60053e+07
[3]	valid_0's l2: 1.39892e+07
[4]	valid_0's l2: 1.2323e+07
[5]	valid_0's l2: 1.09324e+07
[6]	valid_0's l2: 9.76405e+06
[7]	valid_0's l2: 8.78202e+06
[8]	valid_0's l2: 7.95917e+06
[9]	valid_0's l2: 7.27571e+06
[10]	valid_0's l2: 6.70337e+06
[11]	valid_0's l2: 6.22227e+06
[12]	valid_0's l2: 5.80848e+06
[13]	valid_0's l2: 5.45766e+06
[14]	valid_0's l2: 5.15194e+06
[15]	valid_0's l2: 4.88562e+06
[16]	valid_0's l2: 4.66207e+06
[17]	valid_0's l2: 4.4661e+06
[18]	valid_0's l2: 4.29267e+06
[19]	valid_0's l2: 4.14798e+06
[20]	valid_0's l2: 4.02273e+06
[21]	valid_0's l2: 3.90947e+06
[22]	valid_0's l2: 3.8108e+06
[23]	valid_0's l2: 3.73083e+06
[24]	valid_0's l2: 3.65916e+06
[25]	valid_0's l2: 3.59116e+06
[26]	valid_0's l2: 3.53906e+06
[27]	valid_0's l2: 3.48587e+06
[28]	valid_0's l2: 3.43872e+06
[29]	valid_0's l2: 3.39747e+06
[30]	valid_0's l2: 3.35767e+06
[31]	valid_0's l2: 3.3263e+06
[32]	valid_0's l2: 3.29498e+06
[33]	valid_0's l2: 3.

In [32]:
%%time
final_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, num_leaves=63, random_state=42)
final_model.fit(X_train, y_train)

CPU times: user 6.18 s, sys: 32.9 ms, total: 6.21 s
Wall time: 3.11 s


LGBMRegressor(learning_rate=0.05, num_leaves=63, random_state=42)

In [29]:
%%time
predictions = final_model.predict(X_valid)

CPU times: user 830 ms, sys: 0 ns, total: 830 ms
Wall time: 369 ms


## Conclusión final

En este proyecto se evaluaron tres enfoques distintos para predecir el valor de mercado de los vehículos de la plataforma Rusty Bargain, considerando la calidad de predicción (medida en RECM/RMSE), el tiempo de entrenamiento y la velocidad de inferencia (predicción).   

Comparativa de Modelos   
Prueba de cordura (Regresión Lineal):
Alcanzó un RECM de 2606.30, marcando la línea base. Aunque su tiempo de predicción es muy bajo (220 ms), su margen de error es significativamente más alto que el de los modelos basados en árboles, lo que demuestra que los datos tienen relaciones no lineales complejas que la regresión lineal no logra capturar.   

Bosque Aleatorio (Random Forest):
Alcanzó una excelente precisión con un RECM de 1695.17 tras ajustar los hiperparámetros (max_depth=15, n_estimators=100). Sin embargo, demostró ser el modelo más pesado en recursos: requirió el mayor tiempo de entrenamiento (18 s) y el tiempo de predicción más elevado (509 ms).   

LightGBM:
Ofreció la mejor precisión de todo el estudio con un RECM de 1683.67, superando tanto a la regresión lineal como al bosque aleatorio. Destaca por ser el modelo más rápido en entrenar (2.89 s) gracias a su arquitectura optimizada basada en histogramas y manejo nativo de variables categóricas, además de ofrecer una predicción rápida (369 ms).   

Recomendación para la Empresa
Para la implementación en la aplicación de Rusty Bargain, el modelo recomendado es LightGBM por las siguientes razones clave:

Precisión equivalente: Ofrece la menor tasa de error en la estimación de precios (RECM de 1683.67). Aunque el Bosque Aleatorio logra una calidad prácticamente idéntica (1695.17), LightGBM destaca al alcanzar esa misma precisión en una fracción del tiempo.

Eficiencia en entrenamiento: Es más de 6.2 veces más rápido de entrenar que el Bosque Aleatorio (2.89 s vs 18 s), lo que facilitará reentrenar el modelo de forma periódica y ágil a medida que ingresen nuevos datos.

Escalabilidad y velocidad de inferencia: Responde con predicciones en milisegundos (369 ms) y procesa de forma nativa las variables categóricas, simplificando el pipeline de datos en producción.


# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [x]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [x]  Se realizó el análisis de velocidad y calidad de los modelos